# Exploration phase for developing the model

## 1. Imports

In [1]:
import os
from dotenv import load_dotenv
import pyarrow
from pathlib import Path
import boto3
import pandas as pd
import numpy as np

In [2]:
load_dotenv()
os.getenv('AWS_ACCESS_KEY_ID') is not None

True

## 2. Data set loading

In [3]:
s3 = boto3.client('s3')

BUCKET = 'zrive-ds-data'
PREFIX = 'groceries/box_builder_dataset/'

response = s3.list_objects_v2(
    Bucket = BUCKET,
    Prefix = PREFIX
)

key = response['Contents'][0]['Key']
filename = response['Contents'][0]['Key'].split('/')[-1]

local_data_dir = Path('../../data/module3')
local_path = local_data_dir/filename

if local_path.exists():
        print(f'{filename} already exists')
else:
    print(f'Downloading: {filename}')
    s3.download_file(BUCKET,key,local_path)

feature_frame.csv already exists


In [4]:
df_original= pd.read_csv(local_path)
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2880549 entries, 0 to 2880548
Data columns (total 27 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   variant_id                        int64  
 1   product_type                      object 
 2   order_id                          int64  
 3   user_id                           int64  
 4   created_at                        object 
 5   order_date                        object 
 6   user_order_seq                    int64  
 7   outcome                           float64
 8   ordered_before                    float64
 9   abandoned_before                  float64
 10  active_snoozed                    float64
 11  set_as_regular                    float64
 12  normalised_price                  float64
 13  discount_pct                      float64
 14  vendor                            object 
 15  global_popularity                 float64
 16  count_adults                      fl

In [5]:
df_original.head()

,variant_id,product_type,order_id,user_id,created_at,order_date,user_order_seq,outcome,ordered_before,abandoned_before,...,count_children,count_babies,count_pets,people_ex_baby,days_since_purchase_variant_id,avg_days_to_buy_variant_id,std_days_to_buy_variant_id,days_since_purchase_product_type,avg_days_to_buy_product_type,std_days_to_buy_product_type
0,33826472919172,ricepastapulses,2807985930372,3482464092292,2020-10-05 16:46:19,2020-10-05 00:00:00,3,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
1,33826472919172,ricepastapulses,2808027644036,3466586718340,2020-10-05 17:59:51,2020-10-05 00:00:00,2,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
2,33826472919172,ricepastapulses,2808099078276,3481384026244,2020-10-05 20:08:53,2020-10-05 00:00:00,4,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
3,33826472919172,ricepastapulses,2808393957508,3291363377284,2020-10-06 08:57:59,2020-10-06 00:00:00,2,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
4,33826472919172,ricepastapulses,2808429314180,3537167515780,2020-10-06 10:37:05,2020-10-06 00:00:00,3,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618


In [6]:
df_original['vendor'].nunique()

264

In [7]:
df_original['product_type'].nunique()

62

## 3. Data Set cleaning

In [8]:
df = df_original.copy()

### Orders with at least 5 products

In [9]:
df_order_size = df.groupby("order_id")['variant_id'].count().reset_index(name="n_products")
df_order_size.head()

,order_id,n_products
0,2807985930372,608
1,2808027644036,608
2,2808099078276,608
3,2808393957508,611
4,2808429314180,614


In [10]:
(df_order_size['n_products']<5).sum()

0

We find that there are no order_ids with smaller number of items than 5, therefore we do not have to eliminate any order_id from the initial data set, once this is done, order_id can be droped for model training, as it doesn't contain predictive power.

### Variables with date information to date time

This may be useful for be able to split the data into day, month and hour, so the model can learn from it. Also for later scaling, because as strings, scaler will fail

In [11]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['order_hour'] = df['created_at'].dt.hour
df['order_month'] = df['created_at'].dt.month
df['order_dayweek'] = df['created_at'].dt.dayofweek

The variables created at and order_date will be removed from the feature selection, as their information is contained in this columns

### One-hot encoding of categorical variables 

The string columns that we have to encode are vendor and product_type. As we saw in the first part, they have 62 and 324 values, what will generate 386 new columns, not exagerated for our model.

In [12]:
df = pd.get_dummies(df, columns=["product_type","vendor"])

## Model selection and evaluation

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier

from sklearn.metrics import precision_score, recall_score, f1_score


### Data split and scale

We will remove variant_id, order_id and user_id as well as the variables encoded and tranformed into datetime.
- order id was only used for filtering
- variant_id and user_id are integers, so the model would assign coefficients to them as if they were numerical variables, what makes no sense, and hot-encoding them would generate an extreme number of extra-columns.
- Must be also taken into account that information regarding user_id and variant_id is already contained in the rest of the columns, such as user_order_seq, ordered_before, or avs_days_to_buy_variant_id.
- So, i think that the aim of the model is to develop a probability trasformed then into 1-0 based on a threshold for each line of the data set, that can be then related to a (user,variant_id) by another level of the process that uses the whole dataset.

In [14]:
X = df.drop(columns = [
    'outcome',
    'order_id',
    'created_at',
    'order_date',
    'user_id',
    'variant_id'
])
y = df['outcome']

In [15]:
X_temp,X_test,y_temp,y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

X_train,X_val,y_train,y_val = train_test_split(
    X_temp,
    y_temp,
    test_size = 0.25,
    random_state = 42,
    stratify = y_temp
)



In [16]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2880549 entries, 0 to 2880548
Columns: 348 entries, user_order_seq to vendor_zoflora
dtypes: bool(326), float64(18), int32(3), int64(1)
memory usage: 1.3 GB


In [17]:
num_cols = X_train.select_dtypes(include=["int","float"]).columns

In [18]:
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])


Why do we scale?
- Because in classificaton, models like LogisticRegression minimize their loss function using gradient descent. If we do not normalize the variables and some have very different scales, the model may not converge or spend more time and iterations to do so.
- Furthermore, when using penalizations as L1 or L2, as we will, if some variables have dioferent scales, the model might penalize hard the coefficient of the variable with a higer scale just to compensate the high number it is multiplying, making it seem less important when it may not be so. 

Why do we make only .fit on train?
- Because when we make .fit to the scaler, it learns the mean and std of the features.
- If we apply .fit also to validation and test, the model will then have access to this information when training it on the scaled splits, resulting in data leakage

### Model training

In [ ]:
log_model = LogisticRegression(max_iter = 1000)
log_model.fit(X_train,y_train)

In [ ]:
log_l1_model = LogisticRegre